# Retail Sales Forecasting: Case Study 2

In this notebook, we will prepare daily sales data for time series forecasting, apply a simple forecasting method, and evaluate its accuracy.

## Data: retail_daily_sales.csv
- Daily sales for 15 stores from 2022-01-01 to 2024-12-31
- Columns: `date`, `store_id`, `region`, `total_sales`, `promo_flag`
- Data includes promotions, weekly/annual seasonality, outliers, and missing dates

Let's follow good practices and keep it simple!

## 1. Data Preparation Steps for Time Series Forecasting

**Main Steps:**
1. **Load the data**
2. **Parse dates and sort data**
3. **Handle missing dates (fill gaps)**
4. **Deal with outliers**
5. **Feature engineering (create useful columns)**
6. **Aggregate if needed (e.g., by week, store)**

Let's do each step and explain them simply!

In [1]:
# 1.1 Load the data
import pandas as pd
sales = pd.read_csv('retail_daily_sales.csv')
sales.head()

In [2]:
# 1.2 Parse dates and sort data
sales['date'] = pd.to_datetime(sales['date'])
sales = sales.sort_values(['store_id', 'date'])
sales.head()

### 1.3 Handle Missing Dates
Time series models need regular time steps. Let's make sure each store has a row for every date in the range.

In [3]:
# Create a full date range
full_dates = pd.date_range(sales['date'].min(), sales['date'].max())
stores = sales['store_id'].unique()

# Create a multi-index of all store-date combinations
index = pd.MultiIndex.from_product([stores, full_dates], names=['store_id', 'date'])
sales = sales.set_index(['store_id', 'date']).reindex(index)
# Fill missing sales with 0 or interpolate (here, fill with 0 for simplicity)
sales['total_sales'] = sales['total_sales'].fillna(0)
# Fill other columns as needed
sales['region'] = sales['region'].fillna(method='ffill')
sales['promo_flag'] = sales['promo_flag'].fillna(0)
sales = sales.reset_index()
sales.head()

### 1.4 Deal with Outliers
We'll plot and cap extreme sales values (simple approach for beginners).

In [4]:
import matplotlib.pyplot as plt
# Plot sales distribution
sales['total_sales'].plot(kind='hist', bins=50)
plt.title('Sales Distribution')
plt.xlabel('Total Sales')
plt.show()

# Cap sales at the 99th percentile
cap = sales['total_sales'].quantile(0.99)
sales['total_sales'] = sales['total_sales'].clip(upper=cap)
sales['total_sales'].plot(kind='hist', bins=50)
plt.title('Sales Distribution After Capping')
plt.xlabel('Total Sales')
plt.show()

### 1.5 Feature Engineering
Let's add features for day of week, is_weekend, and promo flag (already present).

In [5]:
sales['day_of_week'] = sales['date'].dt.dayofweek
sales['is_weekend'] = sales['day_of_week'].isin([5, 6]).astype(int)
sales.head()

### 1.6 Aggregate Data (Optional)
For simplicity, let's forecast sales for one store. You can aggregate by week if needed.

In [6]:
# Choose one store for demo
store_id = stores[0]
store_sales = sales[sales['store_id'] == store_id].copy()
store_sales.set_index('date', inplace=True)
store_sales['total_sales'].plot(figsize=(14, 4), title=f'Store {store_id} Daily Sales')
plt.show()

## 2. Simple Forecasting Method: Naïve Approach
We will use the naïve method: tomorrow's forecast is today's sales.

We will split the last 30 days as test data.

In [7]:
# Split train/test
train = store_sales.iloc[:-30]
test = store_sales.iloc[-30:]

# Naive forecast: forecast = last observed value
naive_forecast = train['total_sales'].iloc[-1]
test['forecast'] = naive_forecast
test[['total_sales', 'forecast']].plot(figsize=(10, 4), title='Naive Forecast vs Actual')
plt.show()

## 3. Evaluate Forecast Accuracy
Let's calculate MAE (Mean Absolute Error) and RMSE (Root Mean Squared Error).

In [8]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
mae = mean_absolute_error(test['total_sales'], test['forecast'])
rmse = mean_squared_error(test['total_sales'], test['forecast'], squared=False)
print(f'MAE: {mae:.2f}')
print(f'RMSE: {rmse:.2f}')

## 4. Summary
- We prepared daily retail sales data for time series forecasting
- We handled missing dates and outliers, and did simple feature engineering
- We applied the naïve forecasting method
- We evaluated forecast accuracy with MAE and RMSE

**Next steps:** Try more advanced models (moving average, ARIMA, machine learning, etc.) and experiment with more features or stores.